<div align="center">

# WEEK 02

## DATA CLEANING, STANDARDISATION & DATA QUALITY VALIDATION

### CadetX Virtual Work Experience

### Heavy Supplier, Inventory & Warehouse Analytics

</div>

---

## Notebook Overview

This notebook documents the Week 2 data preparation and quality assurance activities for the CadetX Heavy Supplier, Inventory & Warehouse Analytics project.

The objective is to transform, validate, and prepare the 12 source tables stored in the DuckDB warehouse (`warehouse.duckdb`) for reliable analysis and downstream reporting.

## Weekly Tasks

### Task 1: Type Casting and Standardisation

- Standardise data types and date columns.
- Convert warehouse capacity from text to numeric values.
- Extract product dimensions into separate columns.
- Calculate product volume in cubic metres (m³).
- Clean and standardise categorical values.

### Task 2: Stock Position Reconstruction and Reconciliation

The objective is to reconstruct stock positions from `stock_ledger` using IN, OUT and ADJUSTMENT movements, then reconcile the calculated balances against `running_balance` and `inventory_master.current_stock`.

Discrepancies and adjustment treatment will be documented.


### Task 3: Financial Reconciliation

- Validate financial consistency between transaction headers and line items.
- Check GST calculations and transaction totals.
- Reconcile invoices against payments.
- Identify overpayments, unpaid invoices and outstanding balances.
- Recalculate customer purchase values from transaction records.

### Task 4: Referential Integrity and Business-Rule Checks

- Validate foreign-key relationships.
- Check cancelled purchase orders with missing received dates.
- Validate delivery dates against order dates.
- Examine customer regions without corresponding local branches.

### Task 5: Automated Data Quality Checks and Data Export

- Automate validation checks using pytest assertions.
- Ensure checks can run during subsequent data refreshes.
- Export cleaned datasets as Parquet files.
- Store processed datasets in the `data/processed/` directory.+

### Task 6:  Issue Log and Resolution Decisions

Document identified data-quality issues and record an appropriate decision for each issue:

- **Fix:** Correct an identified data-quality problem.
- **Flag:** Retain the issue for further attention.
- **Accept:** Retain a valid or explainable data condition.
- **Exclude:** Remove records that should not be included in analysis.

## Data Source and Technical Environment

| Component | Description |
|---|---|
| Database | `warehouse.duckdb` |
| Database Engine | DuckDB |
| Programming Language | Python |
| Development Environment | Jupyter Notebook |
| Data Manipulation | Pandas |
| Database Querying | SQL / DuckDB |
| Testing Framework | Pytest |
| Processed Data Format | Parquet |
| Source Tables | 12 operational tables |

## Expected Outcomes

By the end of Week 2, the project is expected to produce:

1. Standardised and validated datasets.
2. Reconciled financial records.
3. Verified referential integrity and business rules.
4. A documented data-quality issue log with resolution decisions.
5. Automated and repeatable data-quality checks.
6. Cleaned datasets exported in Parquet format.

## Data Preservation Approach

The original DuckDB database will be preserved as the raw data source. Data cleaning and standardisation will be performed separately, ensuring that the original records remain available for comparison and verification.

All processed datasets will be stored separately in the `data/processed/` directory.

---

**Project:** Heavy Supplier, Inventory & Warehouse Analytics  
**Programme:** CadetX Virtual Work Experience  
**Sprint:** Week 02 – Data Cleaning and Quality Assurance


In [3]:
import duckdb  
import pandas as pd 

con = duckdb.connect("warehouse.duckdb")

## Task 1: Type Casting and Standardisation

### Objective
To convert the `warehouse_capacity` column in the `branches` table from text values into numeric values measured in square feet (sqft).

### Initial Observation
The warehouse capacity values were stored as text, with the unit `sqft` included in each value.

Examples:
- 45230 sqft
- 38750 sqft
- 41890 sqft
- 36210 sqft
- 29880 sqft
- 34120 sqft

### Transformation Performed
The text unit `sqft` was removed from the original values, and the remaining values were converted into integers.

A new column named `warehouse_capacity_sqft` was created while preserving the original `warehouse_capacity` column.

### Outcome
Warehouse capacity values were successfully converted into numeric values, making them suitable for mathematical calculations and further analysis.

### Data Quality Decision
**ACCEPT** – The values followed a consistent unit format, and the conversion was successful.


In [4]:
con.execute("""
SELECT 
    branch_id,
    branch_name,
    warehouse_capacity
FROM branches
""").df()

,branch_id,branch_name,warehouse_capacity
0,DEL001,Delhi Central,45230 sqft
1,PUN001,Pune Distribution,38750 sqft
2,CHN001,Chennai South Hub,41890 sqft
3,HYD001,Hyderabad Logistics,36210 sqft
4,KOL001,Kolkata East Depot,29880 sqft
5,AHM001,Ahmedabad West Hub,34120 sqft


In [6]:
branches_df = con.execute("""
    SELECT *
    FROM branches
""").df()

branches_df["warehouse_capacity_sqft"] = (
    branches_df["warehouse_capacity"]
    .str.replace(" sqft", "", regex=False)
    .astype(int)
)

branches_df[[
    "branch_name",
    "warehouse_capacity",
    "warehouse_capacity_sqft"
]]


,branch_name,warehouse_capacity,warehouse_capacity_sqft
0,Delhi Central,45230 sqft,45230
1,Pune Distribution,38750 sqft,38750
2,Chennai South Hub,41890 sqft,41890
3,Hyderabad Logistics,36210 sqft,36210
4,Kolkata East Depot,29880 sqft,29880
5,Ahmedabad West Hub,34120 sqft,34120


## Product Dimensions: Standardisation

### Objective
To separate the combined product dimensions stored in the `dimensions_cm` column of the `products` table into three individual numeric columns.

### Initial Observation
Product dimensions were stored together as text in the following format:

Example: `10x 20x30`

The three measurements represent:
- Length
- Width
- Height

### Transformation Performed
Regular expressions (regex) were used with Pandas to extract the three measurements.

Three new columns were created:

- `length_cm`
- `width_cm`
- `height_cm`

The extracted values were converted into numeric data types.

The original `dimensions_cm` column was preserved.

### Outcome
Product dimensions were successfully separated into individual numeric columns, making them easier to use in calculations and warehouse analysis.

### Data Quality Decision
**STANDARDISED** – Combined dimension values were separated into individual numeric measurements.


In [5]:
con.execute("""
SELECT 
    product_id,
    product_name,
    dimensions_cm
FROM products
""").df()

,product_id,product_name,dimensions_cm
0,P001,Hydraulic Pump HP-300,40x25x25
1,P002,Engine Oil Filter OF-90,10x8x8
2,P003,Air Filter AF-120,18x12x12
3,P004,Track Chain TC-45,180x40x35
4,P005,Fuel Injector FI-220,20x12x12
5,P006,Alternator ALT-500,30x20x20
6,P007,Hydraulic Hose HH-60,90x4x4
7,P008,Bucket Tooth BT-10,12x6x4
8,P009,Starter Motor SM-700,32x18x18
9,P010,Turbocharger TB-900,40x25x25


In [7]:
products_df = con.execute("""
    SELECT *
    FROM products
""").df()

products_df[["product_id", "product_name", "dimensions_cm"]].head(10)


,product_id,product_name,dimensions_cm
0,P001,Hydraulic Pump HP-300,40x25x25
1,P002,Engine Oil Filter OF-90,10x8x8
2,P003,Air Filter AF-120,18x12x12
3,P004,Track Chain TC-45,180x40x35
4,P005,Fuel Injector FI-220,20x12x12
5,P006,Alternator ALT-500,30x20x20
6,P007,Hydraulic Hose HH-60,90x4x4
7,P008,Bucket Tooth BT-10,12x6x4
8,P009,Starter Motor SM-700,32x18x18
9,P010,Turbocharger TB-900,40x25x25


In [8]:
dimensions = products_df["dimensions_cm"].str.extract(
    r"(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)"
)

dimensions.columns = ["length_cm", "width_cm", "height_cm"]

dimensions = dimensions.astype(float)

products_df = pd.concat([products_df, dimensions], axis=1)

products_df[[
    "product_name",
    "dimensions_cm",
    "length_cm",
    "width_cm",
    "height_cm"
]].head(10)


,product_name,dimensions_cm,length_cm,width_cm,height_cm
0,Hydraulic Pump HP-300,40x25x25,40.0,25.0,25.0
1,Engine Oil Filter OF-90,10x8x8,10.0,8.0,8.0
2,Air Filter AF-120,18x12x12,18.0,12.0,12.0
3,Track Chain TC-45,180x40x35,180.0,40.0,35.0
4,Fuel Injector FI-220,20x12x12,20.0,12.0,12.0
5,Alternator ALT-500,30x20x20,30.0,20.0,20.0
6,Hydraulic Hose HH-60,90x4x4,90.0,4.0,4.0
7,Bucket Tooth BT-10,12x6x4,12.0,6.0,4.0
8,Starter Motor SM-700,32x18x18,32.0,18.0,18.0
9,Turbocharger TB-900,40x25x25,40.0,25.0,25.0


## Product Volume Calculation

### Objective
To calculate the volume of each product in cubic metres (m³) using the standardised length, width and height columns.

### Calculation Method

The volume formula is:

Volume = Length × Width × Height

Since the product dimensions are measured in centimetres, the result must be converted from cubic centimetres (cm³) to cubic metres (m³).

### Conversion Formula

Volume (m³) = (Length × Width × Height) / 1,000,000

### Example

For a product with:

- Length = 10 cm
- Width = 20 cm
- Height = 30 cm

Calculation:

10 × 20 × 30 = 6,000 cm³

6,000 / 1,000,000 = 0.006 m³

### Transformation Performed
A new column named `volume_m3` was created in the products DataFrame.

The calculation used the standardised length, width and height columns.

### Outcome
Product volume was calculated in cubic metres, providing an additional measurement for inventory and warehouse capacity analysis.


In [9]:
products_df["volume_m3"] = (
    products_df["length_cm"] *
    products_df["width_cm"] *
    products_df["height_cm"]
) / 1000000

In [10]:
products_df[[
    "product_name",
    "length_cm",
    "width_cm",
    "height_cm",
    "volume_m3"
]].head(10)

,product_name,length_cm,width_cm,height_cm,volume_m3
0,Hydraulic Pump HP-300,40.0,25.0,25.0,0.025000
1,Engine Oil Filter OF-90,10.0,8.0,8.0,0.000640
2,Air Filter AF-120,18.0,12.0,12.0,0.002592
3,Track Chain TC-45,180.0,40.0,35.0,0.252000
4,Fuel Injector FI-220,20.0,12.0,12.0,0.002880
5,Alternator ALT-500,30.0,20.0,20.0,0.012000
6,Hydraulic Hose HH-60,90.0,4.0,4.0,0.001440
7,Bucket Tooth BT-10,12.0,6.0,4.0,0.000288
8,Starter Motor SM-700,32.0,18.0,18.0,0.010368
9,Turbocharger TB-900,40.0,25.0,25.0,0.025000


## Date Standardisation and Validation

### Objective
To examine date columns across the project database, verify their data types and formats, and identify missing date values.

### Initial Findings
The inspected date columns followed the `YYYY-MM-DD` format.

Example: `2023-08-28`

The inspected columns were already stored using the Date data type. Therefore, additional type conversion was not required for these columns.

### Tables Reviewed

| Table | Date Columns |
|---|---|
| products | last_purchase_date |
| customers | customer_since, last_purchase_date |
| invoices | invoice_date, due_date |
| payments | payment_date |
| purchase_orders_header | order_date, expected_delivery_date, received_date |
| sales_orders_header | order_date, delivery_date |
| stock_ledger | movement_date |


In [11]:
con.execute("""
    SELECT
        product_id,
        product_name,
        last_purchase_date
    FROM products
    LIMIT 20
""").df()

,product_id,product_name,last_purchase_date
0,P001,Hydraulic Pump HP-300,2019-03-15
1,P002,Engine Oil Filter OF-90,2019-07-22
2,P003,Air Filter AF-120,2020-01-10
3,P004,Track Chain TC-45,2020-06-18
4,P005,Fuel Injector FI-220,2020-11-05
5,P006,Alternator ALT-500,2021-02-14
6,P007,Hydraulic Hose HH-60,2021-05-27
7,P008,Bucket Tooth BT-10,2021-08-09
8,P009,Starter Motor SM-700,2021-11-30
9,P010,Turbocharger TB-900,2022-03-12


In [12]:
con.execute("""
    DESCRIBE products
""").df()

,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,category,VARCHAR,YES,None,None,None
3,machine_type,VARCHAR,YES,None,None,None
4,brand,VARCHAR,YES,None,None,None
5,model_compatibility,VARCHAR,YES,None,None,None
6,unit_cost,BIGINT,YES,None,None,None
7,unit_price,BIGINT,YES,None,None,None
8,margin_percentage,DOUBLE,YES,None,None,None
9,gst_rate,BIGINT,YES,None,None,None


In [14]:
products_df["last_purchase_date"].isna().sum()

np.int64(0)

In [15]:
con.execute("""
    SELECT
        customer_id,
        customer_since,
        last_purchase_date
    FROM customers
    LIMIT 20
""").df()

,customer_id,customer_since,last_purchase_date
0,C0001,2024-04-10,2024-04-25
1,C0002,2022-01-19,2024-12-04
2,C0003,2017-02-14,2019-06-11
3,C0004,2017-12-20,2021-12-26
4,C0005,2023-07-27,2024-02-17
5,C0006,2021-10-14,2022-05-10
6,C0007,2019-03-17,2021-04-22
7,C0008,2015-06-24,2017-09-09
8,C0009,2022-04-07,2023-09-12
9,C0010,2015-12-30,2024-03-29


In [17]:
customers_df = con.execute("""
    SELECT *
    FROM customers
""").df()

customers_df[[
    "customer_since",
    "last_purchase_date"]].isna().sum()

customer_since        0
last_purchase_date    0
dtype: int64

In [19]:
con.execute("""
    SELECT
        invoice_id,
        invoice_date,
        due_date
    FROM invoices
    LIMIT 20
""").df()

,invoice_id,invoice_date,due_date
0,INV-325361,2022-05-28,2022-06-27
1,INV-568476,2022-05-08,2022-06-07
2,INV-786913,2023-10-02,2023-11-01
3,INV-697615,2021-10-09,2021-11-08
4,INV-747829,2024-11-04,2024-12-19
5,INV-154762,2022-02-26,2022-03-28
6,INV-159498,2019-10-18,2019-11-02
7,INV-324708,2024-05-30,2024-06-29
8,INV-705737,2021-01-16,2021-01-31
9,INV-156642,2022-02-21,2022-03-23


In [20]:
invoices_df = con.execute("""
    SELECT *
    FROM invoices
""").df()

invoices_df[[
    "invoice_id",
    "due_date"]].isna().sum()

invoice_id    0
due_date      0
dtype: int64

### To Check The Remaining Dates

In [21]:
date_check = con.execute("""
    SELECT
        'payments' AS table_name,
        'payment_date' AS column_name,
        COUNT(*) AS total_records,
        COUNT(payment_date) AS valid_dates,
        COUNT(*) - COUNT(payment_date) AS missing_dates
    FROM payments

    UNION ALL

    SELECT
        'purchase_orders_header',
        'order_date',
        COUNT(*),
        COUNT(order_date),
        COUNT(*) - COUNT(order_date)
    FROM purchase_orders_header

    UNION ALL

    SELECT
        'purchase_orders_header',
        'expected_delivery_date',
        COUNT(*),
        COUNT(expected_delivery_date),
        COUNT(*) - COUNT(expected_delivery_date)
    FROM purchase_orders_header

    UNION ALL

    SELECT
        'purchase_orders_header',
        'received_date',
        COUNT(*),
        COUNT(received_date),
        COUNT(*) - COUNT(received_date)
    FROM purchase_orders_header

    UNION ALL

    SELECT
        'sales_orders_header',
        'order_date',
        COUNT(*),
        COUNT(order_date),
        COUNT(*) - COUNT(order_date)
    FROM sales_orders_header

    UNION ALL

    SELECT
        'sales_orders_header',
        'delivery_date',
        COUNT(*),
        COUNT(delivery_date),
        COUNT(*) - COUNT(delivery_date)
    FROM sales_orders_header

    UNION ALL

    SELECT
        'stock_ledger',
        'movement_date',
        COUNT(*),
        COUNT(movement_date),
        COUNT(*) - COUNT(movement_date)
    FROM stock_ledger
""").df()

date_check

,table_name,column_name,total_records,valid_dates,missing_dates
0,payments,payment_date,19257,19257,0
1,purchase_orders_header,order_date,24000,24000,0
2,purchase_orders_header,expected_delivery_date,24000,24000,0
3,purchase_orders_header,received_date,24000,21630,2370
4,sales_orders_header,order_date,20000,20000,0
5,sales_orders_header,delivery_date,20000,20000,0
6,stock_ledger,movement_date,237230,237230,0


### Missing Date Investigation

The initial checks identified 2,370 missing values in the `received_date` column of the `purchase_orders_header` table.

Further investigation was conducted by grouping purchase orders according to their status.

| PO Status | Total Orders | Received Dates | Missing Dates |
|---|---:|---:|---:|
| Received | 21,630 | 21,630 | 0 |
| Cancelled | 2,370 | 0 | 2,370 |

### Findings
All purchase orders marked as `Received` had valid received dates.

All purchase orders marked as `Cancelled` had missing received dates.

The missing values were consistent with the business process because cancelled purchase orders were not received.

### Data Quality Decision
**ACCEPT** – The 2,370 missing received dates were considered valid business-related missing values. No artificial dates were inserted.

### Overall Outcome
The date review confirmed that the inspected date columns followed a consistent date format and were stored using the Date data type.

The identified missing values in `received_date` were investigated and explained by the cancelled purchase order status.


In [22]:
con.execute("""
    SELECT
        po_status,
        COUNT(*) AS total_orders,
        COUNT(received_date) AS received_dates,
        COUNT(*) - COUNT(received_date) AS missing_dates
    FROM purchase_orders_header
    GROUP BY po_status
    ORDER BY total_orders DESC
""").df()

,po_status,total_orders,received_dates,missing_dates
0,Received,21630,21630,0
1,Cancelled,2370,0,2370


# Categorical Clean-up 


In [23]:
categorical_check = con.execute("""
    SELECT 'branches' AS table_name,
           'warehouse_type' AS column_name,
           CAST(warehouse_type AS VARCHAR) AS original_value,
           TRIM(CAST(warehouse_type AS VARCHAR)) AS cleaned_value,
           COUNT(*) AS frequency
    FROM branches
    GROUP BY warehouse_type

    UNION ALL

    SELECT 'products', 'category',
           CAST(category AS VARCHAR),
           TRIM(CAST(category AS VARCHAR)),
           COUNT(*)
    FROM products
    GROUP BY category

    UNION ALL

    SELECT 'products', 'machine_type',
           CAST(machine_type AS VARCHAR),
           TRIM(CAST(machine_type AS VARCHAR)),
           COUNT(*)
    FROM products
    GROUP BY machine_type

    UNION ALL

    SELECT 'suppliers', 'supplier_type',
           CAST(supplier_type AS VARCHAR),
           TRIM(CAST(supplier_type AS VARCHAR)),
           COUNT(*)
    FROM suppliers
    GROUP BY supplier_type

    UNION ALL

    SELECT 'customers', 'customer_type',
           CAST(customer_type AS VARCHAR),
           TRIM(CAST(customer_type AS VARCHAR)),
           COUNT(*)
    FROM customers
    GROUP BY customer_type

    UNION ALL

    SELECT 'invoices', 'payment_status',
           CAST(payment_status AS VARCHAR),
           TRIM(CAST(payment_status AS VARCHAR)),
           COUNT(*)
    FROM invoices
    GROUP BY payment_status

    UNION ALL

    SELECT 'purchase_orders_header', 'po_status',
           CAST(po_status AS VARCHAR),
           TRIM(CAST(po_status AS VARCHAR)),
           COUNT(*)
    FROM purchase_orders_header
    GROUP BY po_status

    UNION ALL

    SELECT 'sales_orders_header', 'order_status',
           CAST(order_status AS VARCHAR),
           TRIM(CAST(order_status AS VARCHAR)),
           COUNT(*)
    FROM sales_orders_header
    GROUP BY order_status

    UNION ALL

    SELECT 'sales_orders_header', 'sales_channel',
           CAST(sales_channel AS VARCHAR),
           TRIM(CAST(sales_channel AS VARCHAR)),
           COUNT(*)
    FROM sales_orders_header
    GROUP BY sales_channel

    UNION ALL

    SELECT 'payments', 'payment_method',
           CAST(payment_method AS VARCHAR),
           TRIM(CAST(payment_method AS VARCHAR)),
           COUNT(*)
    FROM payments
    GROUP BY payment_method

    UNION ALL

    SELECT 'stock_ledger', 'movement_type',
           CAST(movement_type AS VARCHAR),
           TRIM(CAST(movement_type AS VARCHAR)),
           COUNT(*)
    FROM stock_ledger
    GROUP BY movement_type

""").df()

categorical_check

,table_name,column_name,original_value,cleaned_value,frequency
0,branches,warehouse_type,Regional,Regional,2
1,branches,warehouse_type,Central,Central,1
2,branches,warehouse_type,Branch,Branch,2
3,branches,warehouse_type,Service Hub,Service Hub,1
4,products,category,Engine,Engine,4
5,products,category,Pins & Bushes,Pins & Bushes,1
6,products,category,Cabin,Cabin,1
7,products,category,Filters,Filters,2
8,products,category,Undercarriage,Undercarriage,4
9,products,category,Electrical,Electrical,3


### Identify capitalisation differences

In [24]:
categorical_check["standardised_candidate"] = (
    categorical_check["cleaned_value"].str.lower()
)

categorical_check

,table_name,column_name,original_value,cleaned_value,frequency,standardised_candidate
0,branches,warehouse_type,Regional,Regional,2,regional
1,branches,warehouse_type,Central,Central,1,central
2,branches,warehouse_type,Branch,Branch,2,branch
3,branches,warehouse_type,Service Hub,Service Hub,1,service hub
4,products,category,Engine,Engine,4,engine
5,products,category,Pins & Bushes,Pins & Bushes,1,pins & bushes
6,products,category,Cabin,Cabin,1,cabin
7,products,category,Filters,Filters,2,filters
8,products,category,Undercarriage,Undercarriage,4,undercarriage
9,products,category,Electrical,Electrical,3,electrical


# Final Validation

In [25]:
# Validate warehouse capacity
print("Missing numeric capacities:")

print(branches_df["warehouse_capacity_sqft"].isna().sum())

print("\nWarehouse capacity data type:")

print(branches_df["warehouse_capacity_sqft"].dtype)


Missing numeric capacities:
0

Warehouse capacity data type:
int64


In [26]:
# Validate product dimensions
print("Missing dimensions:")

print(products_df[
    ["length_cm", "width_cm", "height_cm"]
].isna().sum())

print("\nProduct dimensions data types:")

print(products_df[
    ["length_cm", "width_cm", "height_cm"]
].dtypes)


Missing dimensions:
length_cm    0
width_cm     0
height_cm    0
dtype: int64

Product dimensions data types:
length_cm    float64
width_cm     float64
height_cm    float64
dtype: object


In [27]:
# Validate product volume
print("Missing volume values:")

print(products_df["volume_m3"].isna().sum())

print("\nProduct volume data type:")

print(products_df["volume_m3"].dtype)


Missing volume values:
0

Product volume data type:
float64


In [29]:
# Check whether any dimensions failed extraction
products_df[
    products_df[
        ["length_cm", "width_cm", "height_cm"]
    ].isna().any(axis=1)
][[
    "product_id",
    "product_name",
    "dimensions_cm"
]]

,product_id,product_name,dimensions_cm


In [30]:
products_df["volume_m3"].isna().sum()

np.int64(0)

In [31]:
products_df[
    (products_df["length_cm"] <= 0) |
    (products_df["width_cm"] <= 0) |
    (products_df["height_cm"] <= 0)
][["product_id", "product_name", "length_cm", "width_cm", "height_cm"]]

,product_id,product_name,length_cm,width_cm,height_cm


## Task 1: Conclusion – Type Casting and Standardisation

Task 1 was successfully completed. Date fields were reviewed, warehouse capacity was converted to numeric values, product dimensions were extracted, and volume was calculated in cubic metres (m³). Categorical values were also cleaned and standardised.

Validation confirmed that product dimensions and volume calculations contained no missing, zero, or negative values. Missing received dates in cancelled purchase orders were accepted as valid business conditions.

All transformations were performed separately using Python DataFrames, preserving the original `warehouse.duckdb` database.

**Status: COMPLETED**

# Task 2: Stock Positions

## Task 2.1: Stock Movement Inspection

The first step is to examine the stock ledger to understand the recorded movement types, quantity directions, and existing running balances.

This inspection will help establish the correct treatment of IN, OUT, and ADJUSTMENT transactions before reconstructing stock positions.

In [33]:
Movement_summary = con.execute("""
    SELECT
        Movement_type,
        COUNT(*) AS total_records,
        SUM(quantity) AS total_quantity,
        MIN(quantity) AS minimum_quantity,
        MAX(quantity) AS maximum_quantity,

        COUNT(CASE WHEN quantity > 0 THEN 1 END) AS positive_movements,
        COUNT(CASE WHEN quantity < 0 THEN 1 END) AS negative_movements,
        COUNT(CASE WHEN quantity = 0 THEN 1 END) AS zero_movements

    FROM stock_ledger
    GROUP BY movement_type
    ORDER BY movement_type
""").df()

Movement_summary

,movement_type,total_records,total_quantity,minimum_quantity,maximum_quantity,positive_movements,negative_movements,zero_movements
0,ADJUSTMENT,2454,66990.0,5,50,2454,0,0
1,IN,127611,20397795.0,20,300,127611,0,0
2,OUT,107165,1124832.0,1,20,107165,0,0


### Stock Movement Inspection

The `stock_ledger` table was inspected. Three movement types were identified: IN (127,611), OUT (107,165), and ADJUSTMENT (2,454).

All movement quantities were positive. However, adjustment direction could not be established, and discrepancies were observed between movement quantities and recorded balances.

**Status: COMPLETED**


In [35]:
adjustment_check = con.execute("""
    WITH movement_history AS (
        SELECT
            product_id,
            branch_id,
            movement_id,
            movement_date,
            movement_type,
            quantity,
            running_balance,

            LAG(running_balance) OVER (
                PARTITION BY product_id, branch_id
                ORDER BY movement_date, movement_id
            ) AS previous_balance

        FROM stock_ledger
    )

    SELECT
        movement_type,
        COUNT(*) AS total_records,

        SUM(CASE
            WHEN running_balance - previous_balance = quantity
            THEN 1 ELSE 0
        END) AS balance_increased,

        SUM(CASE
            WHEN running_balance - previous_balance = -quantity
            THEN 1 ELSE 0
        END) AS balance_decreased,

        SUM(CASE
            WHEN previous_balance IS NULL
            THEN 1 ELSE 0
        END) AS first_movements

    FROM movement_history
    GROUP BY movement_type
    ORDER BY movement_type
""").df()

adjustment_check

,movement_type,total_records,balance_increased,balance_decreased,first_movements
0,ADJUSTMENT,2454,732.0,757.0,15.0
1,IN,127611,93957.0,27.0,21.0
2,OUT,107165,219.0,85778.0,144.0


In [34]:
Stock_sample = con.execute("""
    SELECT
        Movement_id,
        Product_id,
        Branch_id,
        Movement_type,
        Movement_date,
        Quantity,
        Running_balance
    FROM stock_ledger
    ORDER BY movement_date, movement_id
    LIMIT 30
""").df()

Stock_sample

,movement_id,product_id,branch_id,movement_type,movement_date,quantity,running_balance
0,MOV-ADJ-1492,P026,AHM001,ADJUSTMENT,2019-01-01,38,301
1,MOV-ADJ-2416,P003,KOL001,ADJUSTMENT,2019-01-01,19,253
2,MOV-ADJ-1386,P027,HYD001,ADJUSTMENT,2019-01-02,20,143
3,MOV-ADJ-1640,P009,HYD001,ADJUSTMENT,2019-01-02,40,322
4,MOV-ADJ-1351,P008,PUN001,ADJUSTMENT,2019-01-03,43,43
5,MOV-ADJ-1999,P027,CHN001,ADJUSTMENT,2019-01-03,29,55
6,MOV-ADJ-2140,P027,CHN001,ADJUSTMENT,2019-01-03,6,61
7,MOV-ADJ-605,P029,KOL001,ADJUSTMENT,2019-01-03,25,125
8,MOV-OUT-184737,P020,AHM001,OUT,2019-01-03,3,139
9,MOV-OUT-184738,P025,AHM001,OUT,2019-01-03,15,150


In [36]:
movement_exceptions = con.execute("""
    WITH movement_history AS (
        SELECT
            product_id,
            branch_id,
            movement_id,
            movement_date,
            movement_type,
            quantity,
            running_balance,

            LAG(running_balance) OVER (
                PARTITION BY product_id, branch_id
                ORDER BY movement_date, movement_id
            ) AS previous_balance

        FROM stock_ledger
    )

    SELECT
        movement_id,
        product_id,
        branch_id,
        movement_date,
        movement_type,
        quantity,
        previous_balance,
        running_balance,

        running_balance - previous_balance AS balance_change

    FROM movement_history

    WHERE previous_balance IS NOT NULL
      AND running_balance - previous_balance NOT IN (
          quantity,
          -quantity
      )

    ORDER BY movement_date, movement_id
    LIMIT 30
""").df()

movement_exceptions

,movement_id,product_id,branch_id,movement_date,movement_type,quantity,previous_balance,running_balance,balance_change
0,MOV-OUT-190798,P029,DEL001,2019-01-10,OUT,15,45,83,38
1,MOV-IN-148015,P028,PUN001,2019-01-11,IN,143,276,539,263
2,MOV-IN-77415,P028,PUN001,2019-01-11,IN,120,539,396,-143
3,MOV-IN-9938,P029,DEL001,2019-01-11,IN,169,83,214,131
4,MOV-OUT-256134,P028,PUN001,2019-01-11,OUT,18,396,521,125
5,MOV-OUT-203399,P002,AHM001,2019-01-12,OUT,18,122,84,-38
6,MOV-IN-96009,P002,AHM001,2019-01-13,IN,300,84,422,338
7,MOV-ADJ-1052,P010,HYD001,2019-01-14,ADJUSTMENT,32,132,285,153
8,MOV-IN-152064,P010,HYD001,2019-01-14,IN,121,285,253,-32
9,MOV-ADJ-1794,P013,PUN001,2019-01-15,ADJUSTMENT,10,75,79,4


In [37]:
opening_check = con.execute("""
    SELECT
        COUNT(*) AS total_inventory_records,
        COUNT(opening_stock) AS records_with_opening_stock,
        COUNT(*) - COUNT(opening_stock) AS missing_opening_stock,
        COUNT(current_stock) AS records_with_current_stock
    FROM inventory_master
""").df()

opening_check

,total_inventory_records,records_with_opening_stock,missing_opening_stock,records_with_current_stock
0,180,180,0,180


In [38]:
adjustment_references = con.execute("""
    SELECT
        reference_type,
        COUNT(*) AS total_records
    FROM stock_ledger
    WHERE movement_type = 'ADJUSTMENT'
    GROUP BY reference_type
    ORDER BY total_records DESC
""").df()

adjustment_references

,reference_type,total_records
0,ADJ,2454


In [39]:
adjustment_sample = con.execute("""
    SELECT
        movement_id,
        reference_type,
        reference_id,
        quantity
    FROM stock_ledger
    WHERE movement_type = 'ADJUSTMENT'
    LIMIT 10
""").df()

adjustment_sample

,movement_id,reference_type,reference_id,quantity
0,MOV-ADJ-1519,ADJ,ADJ-1519,38
1,MOV-ADJ-1097,ADJ,ADJ-1097,38
2,MOV-ADJ-524,ADJ,ADJ-524,33
3,MOV-ADJ-920,ADJ,ADJ-920,20
4,MOV-ADJ-1040,ADJ,ADJ-1040,50
5,MOV-ADJ-1050,ADJ,ADJ-1050,5
6,MOV-ADJ-657,ADJ,ADJ-657,22
7,MOV-ADJ-933,ADJ,ADJ-933,31
8,MOV-ADJ-421,ADJ,ADJ-421,13
9,MOV-ADJ-611,ADJ,ADJ-611,12


In [40]:
stock_reconstruction = con.execute("""
    WITH movement_totals AS (
        SELECT
            product_id,
            branch_id,

            SUM(CASE
                WHEN movement_type = 'IN'
                THEN quantity ELSE 0
            END) AS total_in,

            SUM(CASE
                WHEN movement_type = 'OUT'
                THEN quantity ELSE 0
            END) AS total_out,

            SUM(CASE
                WHEN movement_type = 'ADJUSTMENT'
                THEN quantity ELSE 0
            END) AS total_adjustment,

            COUNT(CASE
                WHEN movement_type = 'ADJUSTMENT'
                THEN 1
            END) AS adjustment_records

        FROM stock_ledger
        GROUP BY product_id, branch_id
    )

    SELECT
        i.product_id,
        i.branch_id,
        i.opening_stock,
        COALESCE(m.total_in, 0) AS total_in,
        COALESCE(m.total_out, 0) AS total_out,
        COALESCE(m.total_adjustment, 0) AS total_adjustment,
        COALESCE(m.adjustment_records, 0) AS adjustment_records,

        i.opening_stock
        + COALESCE(m.total_in, 0)
        - COALESCE(m.total_out, 0)
        AS reconstructed_stock_before_adjustments

    FROM inventory_master i

    LEFT JOIN movement_totals m
        ON i.product_id = m.product_id
        AND i.branch_id = m.branch_id

    ORDER BY i.product_id, i.branch_id
""").df()

stock_reconstruction

,product_id,branch_id,opening_stock,total_in,total_out,total_adjustment,adjustment_records,reconstructed_stock_before_adjustments
0,P001,AHM001,236,112918.0,6385.0,271.0,11,106769.0
1,P001,CHN001,187,107266.0,7965.0,411.0,17,99488.0
2,P001,DEL001,196,121433.0,5859.0,298.0,11,115770.0
3,P001,HYD001,226,113622.0,5012.0,448.0,18,108836.0
4,P001,KOL001,263,110844.0,7210.0,219.0,10,103897.0
...,...,...,...,...,...,...,...,...
175,P030,CHN001,125,113447.0,8099.0,262.0,11,105473.0
176,P030,DEL001,184,109238.0,6133.0,368.0,14,103289.0
177,P030,HYD001,162,109601.0,5060.0,494.0,16,104703.0
178,P030,KOL001,199,110216.0,6940.0,512.0,19,103475.0


### Stock Position Reconstruction

Opening stock was available for all 180 inventory records, with no missing values.

Stock was reconstructed using:

**Opening Stock + IN – OUT**

Adjustment quantities were kept separate because their direction was unresolved. Reconstructed balances were calculated for all 180 product-branch combinations.

**Status: COMPLETED (PROVISIONAL)**


In [42]:
Running_balance_check = con.execute("""
    WITH latest_balance AS (
        SELECT
            Product_id,
            Branch_id,
            Running_balance,
            ROW_NUMBER() OVER (
                PARTITION BY product_id, branch_id
                ORDER BY movement_date DESC, movement_id DESC
            ) AS row_num
        FROM stock_ledger
    )

    SELECT
        r.product_id,
        r.branch_id,
        r.reconstructed_stock_before_adjustments,
        l.running_balance,
        r.reconstructed_stock_before_adjustments
            - l.running_balance AS balance_difference

    FROM stock_reconstruction r

    LEFT JOIN latest_balance l
        ON r.product_id = l.product_id
        AND r.branch_id = l.branch_id
        AND l.row_num = 1

    ORDER BY r.product_id, r.branch_id
""").df()

Running_balance_check

,product_id,branch_id,reconstructed_stock_before_adjustments,running_balance,balance_difference
0,P001,AHM001,106769.0,106714,55.0
1,P001,CHN001,99488.0,99373,115.0
2,P001,DEL001,115770.0,115928,-158.0
3,P001,HYD001,108836.0,108772,64.0
4,P001,KOL001,103897.0,103808,89.0
...,...,...,...,...,...
175,P030,CHN001,105473.0,105447,26.0
176,P030,DEL001,103289.0,103131,158.0
177,P030,HYD001,104703.0,104611,92.0
178,P030,KOL001,103475.0,103465,10.0


In [48]:
Balance_summary = Running_balance_check.groupby(lambda _: "All").agg(
    Total_records=("product_id", "count"),
    Matching_records=("balance_difference", lambda x: (x == 0).sum()),
    Mismatching_records=("balance_difference", lambda x: (x != 0).sum()),
    Average_difference=("balance_difference", "mean"),
    Largest_difference=("balance_difference", "max"),
    Smallest_difference=("balance_difference", "min")
)

Balance_summary

,Total_records,Matching_records,Mismatching_records,Average_difference,Largest_difference,Smallest_difference
All,180,1,179,6.333333,355.0,-298.0


### Reconciliation Against Running Balance

The reconstructed stock balances were compared with the latest recorded `running_balance` for each product and branch.

Out of 180 records, only 1 matched exactly, while 179 showed discrepancies. The average difference was 6.33 units, with differences ranging from -298 to +355.

The discrepancies may partly relate to unresolved stock adjustments.

**Status: COMPLETED**



In [49]:
# Compare the balances
inventory_current = con.execute("""
    SELECT
        product_id,
        branch_id,
        current_stock
    FROM inventory_master
""").df()

current_stock_check = stock_reconstruction.merge(
    inventory_current,
    on=["product_id", "branch_id"],
    how="left"
)

current_stock_check["balance_difference"] = (
    current_stock_check["reconstructed_stock_before_adjustments"]
    - current_stock_check["current_stock"]
)

current_stock_check


,product_id,branch_id,opening_stock,total_in,total_out,total_adjustment,adjustment_records,reconstructed_stock_before_adjustments,current_stock,balance_difference
0,P001,AHM001,236,112918.0,6385.0,271.0,11,106769.0,106714,55.0
1,P001,CHN001,187,107266.0,7965.0,411.0,17,99488.0,99373,115.0
2,P001,DEL001,196,121433.0,5859.0,298.0,11,115770.0,115928,-158.0
3,P001,HYD001,226,113622.0,5012.0,448.0,18,108836.0,108772,64.0
4,P001,KOL001,263,110844.0,7210.0,219.0,10,103897.0,103808,89.0
...,...,...,...,...,...,...,...,...,...,...
175,P030,CHN001,125,113447.0,8099.0,262.0,11,105473.0,105447,26.0
176,P030,DEL001,184,109238.0,6133.0,368.0,14,103289.0,103131,158.0
177,P030,HYD001,162,109601.0,5060.0,494.0,16,104703.0,104611,92.0
178,P030,KOL001,199,110216.0,6940.0,512.0,19,103475.0,103465,10.0


In [50]:
# Summarise discrepancies
current_stock_summary = (
    current_stock_check
    .groupby(lambda _: "All")
    .agg(
        Total_records=("product_id", "count"),
        Matching_records=("balance_difference", lambda x: (x == 0).sum()),
        Mismatching_records=("balance_difference", lambda x: (x != 0).sum()),
        Average_difference=("balance_difference", "mean"),
        Largest_difference=("balance_difference", "max"),
        Smallest_difference=("balance_difference", "min")
    )
)

current_stock_summary

,Total_records,Matching_records,Mismatching_records,Average_difference,Largest_difference,Smallest_difference
All,180,1,179,14.811111,644.0,-298.0


## Step 4: Reconciliation Against Current Stock

Reconstructed stock was compared with `inventory_master.current_stock`. Out of 180 records, 1 matched while 179 showed discrepancies. The average difference was 14.81 units.

The results remain provisional because adjustment directions are unresolved.

**Status: COMPLETED (PROVISIONAL)**


In [55]:
adjustment_check = con.execute("""
    WITH movement_history AS (
        SELECT
            product_id,
            branch_id,
            movement_id,
            movement_date,
            movement_type,
            quantity,
            running_balance,
            LAG(running_balance) OVER (
                PARTITION BY product_id, branch_id
                ORDER BY movement_date, movement_id
            ) AS previous_balance
        FROM stock_ledger
    )
    SELECT
        movement_type,
        COUNT(*) AS total_records,
        SUM(CASE
            WHEN running_balance - previous_balance = quantity
            THEN 1 ELSE 0
        END) AS balance_increased,
        SUM(CASE
            WHEN running_balance - previous_balance = -quantity
            THEN 1 ELSE 0
        END) AS balance_decreased,
        SUM(CASE
            WHEN previous_balance IS NULL
            THEN 1 ELSE 0
        END) AS first_movements
    FROM movement_history
    GROUP BY movement_type
    ORDER BY movement_type
""").df()

adjustment_check

,movement_type,total_records,balance_increased,balance_decreased,first_movements
0,ADJUSTMENT,2454,732.0,757.0,15.0
1,IN,127611,93957.0,27.0,21.0
2,OUT,107165,219.0,85778.0,144.0


In [56]:
# Examine unresolved adjustments
adjustment_summary = adjustment_check.copy()

adjustment_summary["unresolved_changes"] = (
    adjustment_summary["total_records"]
    - adjustment_summary["balance_increased"]
    - adjustment_summary["balance_decreased"]
    - adjustment_summary["first_movements"]
)

adjustment_summary

,movement_type,total_records,balance_increased,balance_decreased,first_movements,unresolved_changes
0,ADJUSTMENT,2454,732.0,757.0,15.0,950.0
1,IN,127611,93957.0,27.0,21.0,33606.0
2,OUT,107165,219.0,85778.0,144.0,21024.0


In [66]:
Inventory_current = con.execute("""
    SELECT
        product_id,
        branch_id,
        current_stock
    FROM inventory_master
""").df()

Current_stock_check = stock_reconstruction.merge(
    Inventory_current,
    on=["product_id", "branch_id"],
    how="left"
)

Current_stock_check["balance_difference"] = (
    Current_stock_check["reconstructed_stock_before_adjustments"]-Current_stock_check["current_stock"]
)

Current_stock_check

,product_id,branch_id,opening_stock,total_in,total_out,total_adjustment,adjustment_records,reconstructed_stock_before_adjustments,current_stock,balance_difference
0,P001,AHM001,236,112918.0,6385.0,271.0,11,106769.0,106714,55.0
1,P001,CHN001,187,107266.0,7965.0,411.0,17,99488.0,99373,115.0
2,P001,DEL001,196,121433.0,5859.0,298.0,11,115770.0,115928,-158.0
3,P001,HYD001,226,113622.0,5012.0,448.0,18,108836.0,108772,64.0
4,P001,KOL001,263,110844.0,7210.0,219.0,10,103897.0,103808,89.0
...,...,...,...,...,...,...,...,...,...,...
175,P030,CHN001,125,113447.0,8099.0,262.0,11,105473.0,105447,26.0
176,P030,DEL001,184,109238.0,6133.0,368.0,14,103289.0,103131,158.0
177,P030,HYD001,162,109601.0,5060.0,494.0,16,104703.0,104611,92.0
178,P030,KOL001,199,110216.0,6940.0,512.0,19,103475.0,103465,10.0


In [70]:
running_balance_check = con.execute("""
    WITH latest_balance AS (
        SELECT
            product_id,
            branch_id,
            running_balance,
            ROW_NUMBER() OVER (
                PARTITION BY product_id, branch_id
                ORDER BY movement_date DESC, movement_id DESC
            ) AS row_num
        FROM stock_ledger
    )

    SELECT
        r.product_id,
        r.branch_id,
        r.reconstructed_stock_before_adjustments,
        l.running_balance,
        r.reconstructed_stock_before_adjustments
            - l.running_balance AS balance_difference

    FROM stock_reconstruction r

    LEFT JOIN latest_balance l
        ON r.product_id = l.product_id
        AND r.branch_id = l.branch_id
        AND l.row_num = 1

    ORDER BY r.product_id, r.branch_id
""").df()

running_balance_check

,product_id,branch_id,reconstructed_stock_before_adjustments,running_balance,balance_difference
0,P001,AHM001,106769.0,106714,55.0
1,P001,CHN001,99488.0,99373,115.0
2,P001,DEL001,115770.0,115928,-158.0
3,P001,HYD001,108836.0,108772,64.0
4,P001,KOL001,103897.0,103808,89.0
...,...,...,...,...,...
175,P030,CHN001,105473.0,105447,26.0
176,P030,DEL001,103289.0,103131,158.0
177,P030,HYD001,104703.0,104611,92.0
178,P030,KOL001,103475.0,103465,10.0


In [71]:
# Produce the final reconciliation summary
Final_summary = pd.DataFrame({
    "Reconciliation": [
        "Reconstructed stock vs running_balance",
        "Reconstructed stock vs current_stock"
    ],
    "Total_records": [
        len(running_balance_check),
        len(current_stock_check)
    ],
    "Matching_records": [
        (running_balance_check["balance_difference"] == 0).sum(),
        (current_stock_check["balance_difference"] == 0).sum()
    ],
    "Mismatching_records": [
        (running_balance_check["balance_difference"] != 0).sum(),
        (current_stock_check["balance_difference"] != 0).sum()
    ],
    "Average_difference": [
        Running_balance_check["balance_difference"].mean(),
        Current_stock_check["balance_difference"].mean()
    ],
    "Largest_difference": [
        Running_balance_check["balance_difference"].max(),
        Current_stock_check["balance_difference"].max()
    ],
    "Smallest_difference": [
        Running_balance_check["balance_difference"].min(),
        Current_stock_check["balance_difference"].min()
    ]
})

Final_summary

,Reconciliation,Total_records,Matching_records,Mismatching_records,Average_difference,Largest_difference,Smallest_difference
0,Reconstructed stock vs running_balance,180,1,179,6.333333,355.0,-298.0
1,Reconstructed stock vs current_stock,180,1,179,14.811111,644.0,-298.0


### Task 2: Findings Summary

Stock positions were reconstructed for 180 product-branch combinations. Reconciliation produced 1 match and 179 mismatches against both `running_balance` and `current_stock`.

Of the 2,454 adjustment records, 950 had unresolved balance changes. These discrepancies were flagged for further investigation.

**Status: ANALYSIS COMPLETED — DISCREPANCIES FLAGGED**

## Task 3: Financial Reconciliation

In [73]:
con.execute("""
    DESCRIBE sales_orders_lines
""").df()

,column_name,column_type,null,key,default,extra
0,so_id,VARCHAR,YES,None,None,None
1,line_number,BIGINT,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,quantity,BIGINT,YES,None,None,None
4,unit_price,BIGINT,YES,None,None,None
5,gst_rate,BIGINT,YES,None,None,None
6,line_total,BIGINT,YES,None,None,None
7,gst_amount,DOUBLE,YES,None,None,None
8,line_grand_total,DOUBLE,YES,None,None,None


In [79]:
con.execute(
    "SELECT * FROM sales_orders_lines LIMIT 5"
).df()

,so_id,line_number,product_id,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0
1,SO-990591,2,P026,2,310,18,620,111.6,731.6
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0


In [85]:
line_check = con.execute("""
    SELECT
        so_id,
        product_id,
        quantity,
        unit_price,
        line_total,
        quantity * unit_price AS calculated_line_total,
        line_total - (quantity * unit_price) AS difference

    FROM sales_orders_lines
""").df()
line_check.head(10)

,so_id,product_id,quantity,unit_price,line_total,calculated_line_total,difference
0,SO-990591,P018,19,11300,214700,214700,0
1,SO-990591,P026,2,310,620,620,0
2,SO-990591,P026,3,310,930,930,0
3,SO-990591,P013,9,1450,13050,13050,0
4,SO-990591,P004,3,39800,119400,119400,0
5,SO-990591,P001,9,24500,220500,220500,0
6,SO-990591,P021,3,46200,138600,138600,0
7,SO-990591,P010,11,34900,383900,383900,0
8,SO-990591,P001,10,24500,245000,245000,0
9,SO-990591,P026,16,310,4960,4960,0


In [88]:
order_total = con.execute("""
    SELECT
        so_id,
        SUM(line_total) AS calculated_line_total,
        SUM(gst_amount) AS calculated_gst,
        SUM(line_grand_total) AS calculated_grand_total
    FROM sales_orders_lines
    GROUP BY so_id
    ORDER BY so_id
""").df()
order_total.head(10)

,so_id,calculated_line_total,calculated_gst,calculated_grand_total
0,SO-100070,228400.0,63952.0,292352.0
1,SO-100088,2170410.0,599713.8,2770123.8
2,SO-100155,1224850.0,339333.0,1564183.0
3,SO-100192,219950.0,44246.0,264196.0
4,SO-100220,133560.0,34510.8,168070.8
5,SO-100241,2008600.0,555343.0,2563943.0
6,SO-100259,1589770.0,423318.6,2013088.6
7,SO-100281,22810.0,4105.8,26915.8
8,SO-100340,2086200.0,584136.0,2670336.0
9,SO-100355,4579700.0,1275106.0,5854806.0


In [72]:
con.execute("""
    DESCRIBE sales_orders_header
""").df()

,column_name,column_type,null,key,default,extra
0,so_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,branch_id,VARCHAR,YES,None,None,None
3,order_date,DATE,YES,None,None,None
4,delivery_date,DATE,YES,None,None,None
5,order_status,VARCHAR,YES,None,None,None
6,payment_terms,VARCHAR,YES,None,None,None
7,total_order_value,BIGINT,YES,None,None,None
8,total_gst_amount,DOUBLE,YES,None,None,None
9,grand_total,DOUBLE,YES,None,None,None


In [81]:
con.execute(
    "SELECT * FROM sales_orders_header LIMIT 5"
).df()

,so_id,customer_id,branch_id,order_date,delivery_date,order_status,payment_terms,total_order_value,total_gst_amount,grand_total,sales_channel
0,SO-990591,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale
1,SO-431510,C0031,KOL001,2022-05-01,2022-05-08,Delivered,Advance,2194850,609423.0,2804273.0,Dealer Network
2,SO-495859,C0013,AHM001,2023-10-01,2023-10-02,Delivered,Advance,2620800,733824.0,3354624.0,Online
3,SO-745068,C0382,HYD001,2021-10-04,2021-10-09,Delivered,Advance,1094950,292961.0,1387911.0,Online
4,SO-175713,C0472,CHN001,2024-10-31,2024-11-04,Delivered,Net 45,1250,225.0,1475.0,Counter Sale


In [89]:
header_reconciliation = con.execute("""
WITH line_totals AS (

    SELECT
        so_id,
        SUM(line_total) AS calculated_line_total,
        SUM(gst_amount) AS calculated_gst,
        SUM(line_grand_total) AS calculated_grand_total
    FROM sales_orders_lines
    GROUP BY so_id

)

SELECT
    h.so_id,
    h.grand_total AS header_total,
    l.calculated_grand_total AS line_total,
    h.grand_total - l.calculated_grand_total AS difference

FROM sales_orders_header h

LEFT JOIN line_totals l
    ON h.so_id = l.so_id;
""").df()
header_reconciliation.head(10)

,so_id,header_total,line_total,difference
0,SO-978960,3224426.0,3224426.0,0.0
1,SO-531691,223138.0,223138.0,0.0
2,SO-104814,905088.0,905088.0,0.0
3,SO-164224,937631.6,937631.6,0.0
4,SO-692444,1837657.6,1837657.6,0.0
5,SO-711870,194909.6,194909.6,0.0
6,SO-406994,315011.6,315011.6,0.0
7,SO-785547,2892078.0,2892078.0,0.0
8,SO-891334,1384083.2,1384083.2,0.0
9,SO-351824,2004613.4,2004613.4,0.0


In [90]:
header_reconciliation["difference"].describe()

count    2.000000e+04
mean     1.279841e-12
std      8.790207e-11
min     -9.313226e-10
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.313226e-10
Name: difference, dtype: float64

In [91]:
header_reconciliation["difference"].apply(
    lambda x: "Matching" if abs(x) < 0.01 else "Mismatching").value_counts()

difference
Matching    20000
Name: count, dtype: int64

#### Header vs Line

- Total order:20,000
- Matching:20,000
- Mismatching: 0

Conclusion: All totals matched within tolerance

**Status: PASSED**

### GST Arithmetic

In [92]:
con.execute("""
    SELECT DISTINCT gst_rate
    FROM sales_orders_lines
    GROUP BY gst_rate
""").df()


,gst_rate
0,18
1,28


In [94]:
gst_check = con.execute("""
    SELECT
    so_id,
    product_id,
    line_total,
    gst_rate,
    gst_amount,

    (line_total * gst_rate / 100) AS expected_gst,

    gst_amount - (line_total * gst_rate / 100) AS difference
FROM sales_orders_lines
""").df()

gst_check.head(10)

,so_id,product_id,line_total,gst_rate,gst_amount,expected_gst,difference
0,SO-990591,P018,214700,28,60116.0,60116.0,7.275958e-12
1,SO-990591,P026,620,18,111.6,111.6,0.000000e+00
2,SO-990591,P026,930,18,167.4,167.4,0.000000e+00
3,SO-990591,P013,13050,18,2349.0,2349.0,0.000000e+00
4,SO-990591,P004,119400,28,33432.0,33432.0,0.000000e+00
5,SO-990591,P001,220500,28,61740.0,61740.0,7.275958e-12
6,SO-990591,P021,138600,28,38808.0,38808.0,7.275958e-12
7,SO-990591,P010,383900,28,107492.0,107492.0,1.455192e-11
8,SO-990591,P001,245000,28,68600.0,68600.0,0.000000e+00
9,SO-990591,P026,4960,18,892.8,892.8,0.000000e+00


In [96]:
gst_check["status"] = gst_check["difference"].abs().apply(lambda x: "Matching" if x < 0.01 else "Mismatching")
gst_check["status"].value_counts()

status
Matching    130402
Name: count, dtype: int64

#### GST Reconciliation
- Records: 130402
- Matching: 130402
- Mismatching: 0

**Status: PASSED**

In [98]:
con.execute("DESCRIBE invoices").df()

,column_name,column_type,null,key,default,extra
0,invoice_id,VARCHAR,YES,None,None,None
1,so_id,VARCHAR,YES,None,None,None
2,customer_id,VARCHAR,YES,None,None,None
3,branch_id,VARCHAR,YES,None,None,None
4,invoice_date,DATE,YES,None,None,None
5,due_date,DATE,YES,None,None,None
6,total_order_value,BIGINT,YES,None,None,None
7,total_gst_amount,DOUBLE,YES,None,None,None
8,grand_total,DOUBLE,YES,None,None,None
9,payment_status,VARCHAR,YES,None,None,None


In [99]:
con.execute("DESCRIBE payments").df()

,column_name,column_type,null,key,default,extra
0,payment_id,VARCHAR,YES,None,None,None
1,invoice_id,VARCHAR,YES,None,None,None
2,payment_date,DATE,YES,None,None,None
3,payment_amount,DOUBLE,YES,None,None,None
4,payment_method,VARCHAR,YES,None,None,None


In [107]:
payment_check = con.execute("""
SELECT
    i.invoice_id,
    i.grand_total,

COALESCE(SUM(p.payment_amount),0) AS total_paid

FROM invoices i
LEFT JOIN payments p
    ON i.invoice_id = p.invoice_id
    GROUP BY 
    i.invoice_id,
    i.grand_total;

""").df()
payment_check.head(10)

,invoice_id,grand_total,total_paid
0,INV-697615,1387911.0,1387911.00
1,INV-154762,1050598.0,1050598.00
2,INV-705737,1771642.8,1771642.80
3,INV-756381,263567.6,263567.60
4,INV-963160,1096536.0,650462.84
5,INV-576405,1580574.0,451238.85
6,INV-620681,9408.0,9408.00
7,INV-165894,819004.6,819004.60
8,INV-853902,3618774.4,3618774.40
9,INV-572541,1375679.0,1375679.00


In [108]:
payment_check['outstanding_balance'] = (
    payment_check['grand_total'] - payment_check['total_paid']
)

payment_check['payment_result'] = payment_check[
    'outstanding_balance'
].apply(
    lambda x: 'Overpaid' if x < -0.01
    else 'Unpaid' if x > 0.01
    else 'Fully Paid'
)

display(payment_check.head(10))

,invoice_id,grand_total,total_paid,outstanding_balance,payment_result
0,INV-697615,1387911.0,1387911.00,0.00,Fully Paid
1,INV-154762,1050598.0,1050598.00,0.00,Fully Paid
2,INV-705737,1771642.8,1771642.80,0.00,Fully Paid
3,INV-756381,263567.6,263567.60,0.00,Fully Paid
4,INV-963160,1096536.0,650462.84,446073.16,Unpaid
5,INV-576405,1580574.0,451238.85,1129335.15,Unpaid
6,INV-620681,9408.0,9408.00,0.00,Fully Paid
7,INV-165894,819004.6,819004.60,0.00,Fully Paid
8,INV-853902,3618774.4,3618774.40,0.00,Fully Paid
9,INV-572541,1375679.0,1375679.00,0.00,Fully Paid


In [109]:
payment_check["payment_result"].value_counts()

payment_result
Fully Paid    12170
Unpaid         5499
Overpaid        364
Name: count, dtype: int64

#### Invoices vs Payments

- Fully Paid: 12170
- Unpaid: 5499
- Overpaid: 364

**Status: PASSED**

In [110]:
con.execute("DESCRIBE customers").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_type,VARCHAR,YES,None,None,None
2,industry_segment,VARCHAR,YES,None,None,None
3,city,VARCHAR,YES,None,None,None
4,state,VARCHAR,YES,None,None,None
5,pincode,BIGINT,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,branch_id,VARCHAR,YES,None,None,None
8,credit_limit,BIGINT,YES,None,None,None
9,current_balance,BIGINT,YES,None,None,None


In [112]:
con.execute("DESCRIBE sales_orders_header").df()

,column_name,column_type,null,key,default,extra
0,so_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,branch_id,VARCHAR,YES,None,None,None
3,order_date,DATE,YES,None,None,None
4,delivery_date,DATE,YES,None,None,None
5,order_status,VARCHAR,YES,None,None,None
6,payment_terms,VARCHAR,YES,None,None,None
7,total_order_value,BIGINT,YES,None,None,None
8,total_gst_amount,DOUBLE,YES,None,None,None
9,grand_total,DOUBLE,YES,None,None,None


In [113]:
customer_reconciliation = con.execute("""

    WITH calculated_purchases AS (
        SELECT
            customer_id,
            SUM(grand_total) AS calculated_purchase_value
        FROM sales_orders_header
        GROUP BY customer_id
    )

    SELECT
        c.customer_id,
        c.total_purchase_value AS stored_purchase_value,

        COALESCE(
            cp.calculated_purchase_value, 0
        ) AS calculated_purchase_value,

        c.total_purchase_value -
        COALESCE(cp.calculated_purchase_value, 0)
        AS difference

    FROM customers c

    LEFT JOIN calculated_purchases cp
        ON c.customer_id = cp.customer_id

""").df()

customer_reconciliation.head(10)

,customer_id,stored_purchase_value,calculated_purchase_value,difference
0,C0288,1107835,107119652.2,-106011817.2
1,C0463,4726488,58325024.0,-53598536.0
2,C0100,1486297,50958417.2,-49472120.2
3,C0489,4594315,78799228.8,-74204913.8
4,C0325,3922336,56775486.8,-52853150.8
5,C0246,1740268,78316881.4,-76576613.4
6,C0349,4461026,65933267.4,-61472241.4
7,C0224,494766,58349595.8,-57854829.8
8,C0019,1569218,54871093.4,-53301875.4
9,C0010,3435041,69405822.8,-65970781.8


In [115]:
customer_reconciliation['status'] = customer_reconciliation[
    'difference'
].abs().apply(lambda x: "Matching" if x < 0.01 else "Mismatching")

customer_reconciliation['status'].value_counts()

status
Mismatching    500
Name: count, dtype: int64

In [116]:
len(customer_reconciliation)

500

#### Customer Purchase Reconciliation

- Customers : 500
- Matching: 0
- Mismatching: 500

**Status: FLAGGED**

### Task 4: Referential Integrity and Business-Rule Checks

In [118]:
# Define candidate foreign key relationships

fk_relationships = [
    ("branches", "branch_id", "inventory_master", "branch_id"),
    ("products", "product_id", "inventory_master", "product_id"),
    ("suppliers", "supplier_id", "purchase_orders_header", "supplier_id"),
    ("branches", "branch_id", "purchase_orders_header", "branch_id"),
    ("purchase_orders_header", "po_id", "purchase_orders_lines", "po_id"),
    ("products", "product_id", "purchase_orders_lines", "product_id"),
    ("customers", "customer_id", "sales_orders_header", "customer_id"),
    ("branches", "branch_id", "sales_orders_header", "branch_id"),
    ("sales_orders_header", "so_id", "sales_orders_lines", "so_id"),
    ("products", "product_id", "sales_orders_lines", "product_id"),
    ("sales_orders_header", "so_id", "invoices", "so_id"),
    ("customers", "customer_id", "invoices", "customer_id"),
    ("branches", "branch_id", "invoices", "branch_id"),
    ("invoices", "invoice_id", "payments", "invoice_id"),
    ("products", "product_id", "stock_ledger", "product_id"),
    ("branches", "branch_id", "stock_ledger", "branch_id"),
    ("branches", "branch_id", "customers", "branch_id")
]

# Store validation results
fk_results = []

# Check each relationship
for parent_table, parent_key, child_table, child_key in fk_relationships:

    query = f"""
        SELECT
            COUNT(*) AS total_records,

            COUNT(*) FILTER (
                WHERE c.{child_key} IS NULL
            ) AS null_keys,

            COUNT(*) FILTER (
                WHERE c.{child_key} IS NOT NULL
                AND NOT EXISTS (
                    SELECT 1
                    FROM {parent_table} p
                    WHERE p.{parent_key} = c.{child_key}
                )
            ) AS orphan_records

        FROM {child_table} c
    """

    result = con.execute(query).fetchone()

    fk_results.append({
        "Parent_Table": parent_table,
        "Child_Table": child_table,
        "Parent_Key": parent_key,
        "Child_Key": child_key,
        "Total_Records": result[0],
        "Null_Keys": result[1],
        "Orphan_Records": result[2]
    })

# Create summary DataFrame
fk_summary = pd.DataFrame(fk_results)

fk_summary

,Parent_Table,Child_Table,Parent_Key,Child_Key,Total_Records,Null_Keys,Orphan_Records
0,branches,inventory_master,branch_id,branch_id,180,0,0
1,products,inventory_master,product_id,product_id,180,0,0
2,suppliers,purchase_orders_header,supplier_id,supplier_id,24000,0,0
3,branches,purchase_orders_header,branch_id,branch_id,24000,0,0
4,purchase_orders_header,purchase_orders_lines,po_id,po_id,155495,0,0
5,products,purchase_orders_lines,product_id,product_id,155495,0,0
6,customers,sales_orders_header,customer_id,customer_id,20000,0,0
7,branches,sales_orders_header,branch_id,branch_id,20000,0,0
8,sales_orders_header,sales_orders_lines,so_id,so_id,130402,0,0
9,products,sales_orders_lines,product_id,product_id,130402,0,0


In [120]:
print(fk_summary.to_string(index=False))

          Parent_Table            Child_Table  Parent_Key   Child_Key  Total_Records  Null_Keys  Orphan_Records
              branches       inventory_master   branch_id   branch_id            180          0               0
              products       inventory_master  product_id  product_id            180          0               0
             suppliers purchase_orders_header supplier_id supplier_id          24000          0               0
              branches purchase_orders_header   branch_id   branch_id          24000          0               0
purchase_orders_header  purchase_orders_lines       po_id       po_id         155495          0               0
              products  purchase_orders_lines  product_id  product_id         155495          0               0
             customers    sales_orders_header customer_id customer_id          20000          0               0
              branches    sales_orders_header   branch_id   branch_id          20000          0         

In [121]:
# Candidate primary keys for all 12 tables

primary_keys = {
    "branches": ["branch_id"],
    "products": ["product_id"],
    "inventory_master": ["product_id", "branch_id"],
    "suppliers": ["supplier_id"],
    "customers": ["customer_id"],
    "invoices": ["invoice_id"],
    "payments": ["payment_id"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "stock_ledger": ["movement_id"]
}

# Store validation results
pk_results = []

# Validate each candidate primary key
for table, keys in primary_keys.items():

    key_columns = ", ".join(keys)

    query = f"""
        SELECT
            COUNT(*) AS total_records,

            COUNT(*) FILTER (
                WHERE {" OR ".join(f"{key} IS NULL" for key in keys)}
            ) AS null_records,

            COUNT(*) - COUNT(DISTINCT ({key_columns}))
                AS duplicate_records

        FROM {table}
    """

    result = con.execute(query).fetchone()

    pk_results.append({
        "Table": table,
        "Candidate_PK": ", ".join(keys),
        "Total_Records": result[0],
        "Null_Records": result[1],
        "Duplicate_Records": result[2]
    })

# Display results
pk_summary = pd.DataFrame(pk_results)

pk_summary

,Table,Candidate_PK,Total_Records,Null_Records,Duplicate_Records
0,branches,branch_id,6,0,0
1,products,product_id,30,0,0
2,inventory_master,"product_id, branch_id",180,0,0
3,suppliers,supplier_id,8,0,0
4,customers,customer_id,500,0,0
5,invoices,invoice_id,18033,0,197
6,payments,payment_id,19257,0,202
7,purchase_orders_header,po_id,24000,0,0
8,purchase_orders_lines,"po_id, line_number",155495,0,0
9,sales_orders_header,so_id,20000,0,0


In [122]:
# Tables and candidate primary keys to investigate

duplicate_tables = {
    "invoices": "invoice_id",
    "payments": "payment_id"
}

# Store duplicate records
duplicate_details = {}

for table, key in duplicate_tables.items():

    query = f"""
        WITH duplicate_keys AS (
            SELECT
                {key},
                COUNT(*) AS occurrences

            FROM {table}

            GROUP BY {key}

            HAVING COUNT(*) > 1
        )

        SELECT
            t.*,
            d.occurrences AS duplicate_count

        FROM {table} t

        JOIN duplicate_keys d
            ON t.{key} = d.{key}

        ORDER BY t.{key}
    """

    duplicate_details[table] = con.execute(query).df()

    print(f"\n{table.upper()} DUPLICATES")
    display(duplicate_details[table])


INVOICES DUPLICATES


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status,duplicate_count
0,INV-105711,SO-549003,C0191,KOL001,2024-12-15,2025-01-14,529940,146609.2,676549.2,Paid,2
1,INV-105711,SO-901766,C0208,KOL001,2021-11-20,2022-01-04,1424270,368413.6,1792683.6,Paid,2
2,INV-106596,SO-807417,C0281,KOL001,2024-01-25,2024-03-25,40100,7218.0,47318.0,Paid,2
3,INV-106596,SO-399413,C0409,PUN001,2021-06-16,2021-07-31,1204730,336001.4,1540731.4,Paid,2
4,INV-111054,SO-316263,C0313,AHM001,2019-08-02,2019-10-01,310,55.8,365.8,Paid,2
...,...,...,...,...,...,...,...,...,...,...,...
388,INV-973856,SO-844752,C0063,DEL001,2021-03-14,2021-04-13,26100,4698.0,30798.0,Paid,2
389,INV-984419,SO-243958,C0486,PUN001,2020-10-08,2020-11-07,1315710,366147.8,1681857.8,Paid,2
390,INV-984419,SO-891909,C0449,KOL001,2019-08-09,2019-09-23,2285640,625115.2,2910755.2,Paid,2
391,INV-987166,SO-595551,C0216,KOL001,2019-04-07,2019-05-07,389440,108179.2,497619.2,Paid,2



PAYMENTS DUPLICATES


,payment_id,invoice_id,payment_date,payment_amount,payment_method,duplicate_count
0,PAY-102690,INV-826669,2021-04-23,3837051.40,Credit Card,2
1,PAY-102690,INV-196859,2021-12-29,520518.80,Cash,2
2,PAY-105592,INV-323176,2022-07-10,1019902.14,Cheque,2
3,PAY-105592,INV-963449,2023-02-23,2123180.75,Cash,2
4,PAY-112476,INV-374653,2024-05-16,1724913.40,Cash,2
...,...,...,...,...,...,...
398,PAY-993408,INV-862026,2023-01-12,1519942.80,Bank Transfer,2
399,PAY-996761,INV-913366,2022-11-03,427.36,Credit Card,2
400,PAY-996761,INV-231558,2019-06-12,4481897.00,Cheque,2
401,PAY-997771,INV-572183,2020-09-18,1073744.80,Cash,2


In [123]:
# Classify duplicate records in invoices and payments

for table, key in duplicate_tables.items():

    df = duplicate_details[table].copy()

    # Exclude the helper column
    original_columns = [
        col for col in df.columns
        if col != "duplicate_count"
    ]

    exact_duplicates = df.duplicated(
        subset=original_columns,
        keep=False
    ).sum()

    duplicate_ids = df[key].nunique()

    print(f"\n{table.upper()} SUMMARY")
    print("Duplicate IDs:", duplicate_ids)
    print("Records involved:", len(df))
    print("Exact duplicate rows:", exact_duplicates)
    print("Non-exact duplicate rows:",
          len(df) - exact_duplicates)

    # Display duplicate ID groups
    display(
        df.groupby(key)
          .size()
          .reset_index(name="Occurrences")
          .query("Occurrences > 1")
          .head(10)
    )


INVOICES SUMMARY
Duplicate IDs: 196
Records involved: 393
Exact duplicate rows: 0
Non-exact duplicate rows: 393


,invoice_id,Occurrences
0,INV-105711,2
1,INV-106596,2
2,INV-111054,2
3,INV-112235,2
4,INV-112378,2
5,INV-124878,2
6,INV-128338,2
7,INV-134655,2
8,INV-137158,2
9,INV-143035,2



PAYMENTS SUMMARY
Duplicate IDs: 201
Records involved: 403
Exact duplicate rows: 0
Non-exact duplicate rows: 403


,payment_id,Occurrences
0,PAY-102690,2
1,PAY-105592,2
2,PAY-112476,2
3,PAY-115474,2
4,PAY-115921,2
5,PAY-117803,2
6,PAY-144401,2
7,PAY-146634,2
8,PAY-151548,2
9,PAY-155156,2


In [124]:
# Recalculate invoice duplicate classification

df = duplicate_details["invoices"].copy()

original_columns = [
    col for col in df.columns
    if col != "duplicate_count"
]

exact = df.duplicated(
    subset=original_columns,
    keep=False
).sum()

total = len(df)

print("INVOICE DUPLICATE VERIFICATION")
print("Records involved:", total)
print("Exact duplicate rows:", exact)
print("Non-exact duplicate rows:", total - exact)
print("Unique duplicate IDs:", df["invoice_id"].nunique())

INVOICE DUPLICATE VERIFICATION
Records involved: 393
Exact duplicate rows: 0
Non-exact duplicate rows: 393
Unique duplicate IDs: 196


In [132]:
# Examine frequency of duplicated identifiers
for table, key in duplicate_tables.items():

    print(f"\n{table.upper()} DUPLICATE FREQUENCY")

    Frequency = (
        duplicate_details[table]
        .groupby(key)
        .size()
        .value_counts()
        .sort_index()
        .reset_index()
    )

    Frequency.columns = [
        "Number_of_Occurrences",
        "Number_of_IDs"
    ]

Frequency


INVOICES DUPLICATE FREQUENCY

PAYMENTS DUPLICATE FREQUENCY


,Number_of_Occurrences,Number_of_IDs
0,2,200
1,3,1
